# 02 — Turn causal assumptions into an estimate

Notebook 00 created the observational data. Notebook 01 summarized what could be seen from those data and produced a DAG worksheet.

Notebook 02 now asks a different question:

> **What causal structure do we believe generated those associations?**

You will build a DAG, identify an estimand with DoWhy, estimate the treatment effect, run refutation checks, and finally reveal the synthetic ground-truth DAG.

### Tutorial path

00 Data preparation → 01 Correlational analysis → **02 Causal inference**


## 1. Configure the causal analysis

The data and worksheet paths are project-relative.

Do not open `synthetic_ground_truth_edges.csv` yet if you want to attempt the DAG exercise first.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from dowhy import CausalModel

from src.causal_graph_ui import CausalGraphBuilder
from src.causal_tutorial_utils import (
    check_dowhy_version,
    load_analysis_data,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_path": "data/dag_worksheet.csv",
        "ground_truth_dag": "data/synthetic_ground_truth_edges.csv",
        "treatment_column": "treatment",
        "outcome_column": "output",
        "covariate_columns": [
            "code_number_tokens",
            "code_complexity",
            "code_num_identifiers",
            "code_num_strings",
            "developer_experience",
            "rollout_eligibility",
            "noise_feature",
            "docstring_detail_score",
            "review_flag",
        ],
        "graph_palette": "husl",
        "graph_edge_opacity": 0.35,
        "refuter_simulations": 50,
    }

params = default_params()

print("DoWhy version:", check_dowhy_version("0.14"))
params


## 2. Load the observed data

At this point we still use only the observed table. The synthetic ground truth remains hidden.


In [ ]:
(
    analysis_df,
    treatment,
    outcome,
    covariates,
    excluded_covariates,
) = load_analysis_data(params)

print(f"Rows: {len(analysis_df):,}")
print(f"Treatment prevalence: {analysis_df[treatment].mean():.3f}")
print(f"Covariates available for the graph: {len(covariates)}")
print(f"Excluded constant covariates: {excluded_covariates or 'none'}")

analysis_df.head()


## 3. Review your DAG worksheet

The worksheet contains statistical clues from notebook 01 and space for causal reasoning.

Focus on the last three columns. They should reflect mechanisms and temporal order—not just the largest correlations.


In [ ]:
worksheet_path = Path(params["dag_worksheet_path"])

if worksheet_path.exists():
    dag_worksheet = pd.read_csv(worksheet_path)
    display(dag_worksheet)
else:
    dag_worksheet = None
    print(
        "DAG worksheet not found. Run "
        "01_correlational_analysis.ipynb first."
    )


## 4. Build the DAG

The graph begins with the causal question itself:

`treatment → output`

Add other edges only when you can justify a causal mechanism.

Pay particular attention to variables that appeared associated with both treatment and outcome in notebook 01. That pattern alone does not tell you whether the variable is:

- a common cause;
- a mediator;
- a collider;
- an instrument candidate;
- a proxy; or
- irrelevant.

The graph editor infers descriptive structural roles **after** you draw the arrows.


In [ ]:
graph_builder = CausalGraphBuilder(
    data_columns=analysis_df.columns,
    treatment=treatment,
    outcome=outcome,
    covariates=covariates,
    palette=params["graph_palette"],
    edge_opacity=params["graph_edge_opacity"],
)

graph_builder.display()


### Freeze your proposed DAG

Click **Use this DAG for analysis**, then run the cell below.

Do this before revealing the synthetic ground truth.


In [ ]:
analysis_dag = graph_builder.get_frozen_graph()

print(
    f"Using DAG with {analysis_dag.number_of_nodes()} nodes "
    f"and {analysis_dag.number_of_edges()} edges."
)


## 5. Create the DoWhy model

The causal model combines the observed data with the assumptions encoded by your graph.

This is the point where we leave purely correlational analysis.


In [ ]:
causal_model = CausalModel(
    data=analysis_df,
    treatment=treatment,
    outcome=outcome,
    graph=analysis_dag,
)

print("DoWhy causal model created from the frozen DAG.")


## 6. Identify the effect

Identification asks whether the Average Treatment Effect can be expressed using observed quantities under the DAG assumptions.

It answers **what should be estimated**, before choosing **how to estimate it**.


In [ ]:
identified_estimand = causal_model.identify_effect(
    proceed_when_unidentifiable=False
)

print(identified_estimand)


## 7. Estimate the Average Treatment Effect

We use inverse propensity-score weighting.

For the synthetic example, notebook 00 also computed the true total ATE. Do not use that value to choose the DAG; use it later to evaluate how well the causal workflow recovered the known effect.


In [ ]:
estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
    control_value=0,
    treatment_value=1,
    method_params={{
        "min_ps_score": 0.05,
        "max_ps_score": 0.95,
        "weighting_scheme": "ips_weight",
    }},
)

print(estimate)
print(f"\nEstimated ATE: {float(estimate.value):.4f}")


## 8. Run refutation checks

Refuters test selected forms of fragility. They do not prove that your DAG is correct.

### Placebo treatment

After treatment is randomly permuted, the placebo effect should generally be near zero.


In [ ]:
placebo_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(placebo_refutation)


### Random common cause

Adding an irrelevant random variable should not materially change a stable estimate.


In [ ]:
random_common_cause_refutation = causal_model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
    num_simulations=params["refuter_simulations"],
    random_seed=RANDOM_SEED,
)

print(random_common_cause_refutation)


# Reveal: compare your DAG with the synthetic ground truth

Now reveal the DAG used to generate the teaching data.

This is the most important comparison in the tutorial. Look back at notebook 01 and ask:

- Which true confounders were easy to spot?
- Did the mediator look like a confounder from correlation alone?
- Did the collider look important?
- Did the instrument candidate show some association with outcome through treatment?
- Did the proxy-like measurement tempt you to add an unnecessary edge?
- Was the irrelevant noise feature correctly ignored?


In [ ]:
ground_truth = pd.read_csv(params["ground_truth_dag"])
display(ground_truth)

true_edges = set(
    zip(
        ground_truth["source"],
        ground_truth["target"],
    )
)
proposed_edges = set(analysis_dag.edges())

missing_edges = sorted(true_edges - proposed_edges)
extra_edges = sorted(proposed_edges - true_edges)

print("\nEdges in the synthetic DAG but missing from your DAG:")
print(missing_edges or "none")

print("\nEdges in your DAG but not in the synthetic DAG:")
print(extra_edges or "none")


## Final interpretation

The tutorial deliberately gives several variables misleadingly similar statistical signatures.

That is the lesson:

**Notebook 00:** defines what was observed and, for this synthetic exercise, creates the hidden causal structure.

**Notebook 01:** shows what association alone can reveal.

**Notebook 02:** requires causal assumptions, then lets you compare those assumptions with a known data-generating DAG.

A good causal workflow does not turn the strongest correlations into adjustment variables. It asks which variables are genuine pre-treatment common causes of treatment and outcome, then checks whether the chosen estimand and estimator follow from that structure.
